In [1]:
import pandas as pd
import numpy as np
import torch

In [3]:
url = "https://huggingface.co/datasets/yandex/yambda/resolve/main/sequential/50m/listens.parquet"
df = pd.read_parquet(url)
print(f"Загружено {len(df)} пользователей")
df.head()

Загружено 9238 пользователей


,uid,timestamp,item_id,is_organic,played_ratio_pct,track_length_seconds
0,100,"[39420, 39420, 39625, 40110, 40360, 40380, 406...","[8326270, 1441281, 286361, 732449, 3397170, 78...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[100, 100, 100, 100, 46, 100, 100, 100, 100, 1...","[170, 105, 185, 240, 130, 205, 205, 145, 95, 2..."
1,200,"[14329075, 14329075, 14329075, 14329075, 14329...","[3285270, 5253582, 9155883, 6113313, 1142869, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[9, 28, 42, 32, 0, 0, 5, 9, 100, 100, 100, 6, ...","[170, 170, 205, 155, 215, 185, 205, 195, 165, ..."
2,300,"[54090, 54100, 54110, 54120, 54125, 54135, 541...","[618910, 8793425, 8842650, 5806319, 8495320, 9...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[2, 4, 4, 5, 1, 5, 7, 1, 2, 2, 10, 6, 1, 23, 6...","[270, 130, 225, 210, 300, 210, 190, 340, 240, ..."
3,500,"[22695440, 22695690, 22695715, 22696095, 22696...","[6417502, 6896222, 6896938, 6950641, 7981221, ...","[0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, ...","[100, 37, 10, 37, 100, 100, 11, 13, 20, 100, 1...","[225, 210, 240, 165, 150, 245, 135, 165, 275, ..."
4,600,"[1329190, 1329405, 1329690, 1329950, 1330185, ...","[8077497, 1865247, 95237, 4624864, 8332575, 44...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[100, 100, 100, 100, 100, 100, 100, 100, 100, ...","[245, 215, 235, 260, 250, 280, 180, 195, 155, ..."


In [26]:
def filter_listen_threshold(df, listen_threshold=50):
  # фильтруем пользователей по порогу в 50 процентов прослушиваний
  filtered_data = []
  for _, row in df.iterrows():
    items = np.array(row['item_id'])
    ratios = np.array(row['played_ratio_pct'])
    mask = ratios >= listen_threshold
    # если есть ХОТЬ ОДИН трек с ≥50% оставляем пользователя
    if mask.any():
      filtered_data.append({
      'uid': row['uid'],
      'item_id': items[mask].tolist(),
      'played_ratio_pct': ratios[mask].tolist()
          })
  df_filtered = pd.DataFrame(filtered_data)
  print(f"После фильтрации: {len(df_filtered)} пользователей")
  return df_filtered

In [24]:
df_filtered = filter_listen_threshold(df, listen_threshold=50)

После фильтрации: 9209 пользователей


In [25]:
df_filtered.head()

,uid,item_id,played_ratio_pct
0,100,"[8326270, 1441281, 286361, 732449, 7849270, 14...","[100, 100, 100, 100, 100, 100, 100, 100, 100, ..."
1,200,"[173016, 8420846, 8085657, 2999079, 299843, 96...","[100, 100, 100, 97, 81, 100, 100, 100, 100, 10..."
2,300,"[6961827, 1926141, 6442688, 6442688, 5196757, ...","[56, 76, 100, 100, 62, 58, 100, 66, 55, 76, 10..."
3,500,"[6417502, 7981221, 4119982, 8350060, 2943786, ...","[100, 100, 100, 100, 100, 100, 100, 100, 100, ..."
4,600,"[8077497, 1865247, 95237, 4624864, 8332575, 44...","[100, 100, 100, 100, 100, 100, 100, 100, 100, ..."
